# Music Annotation Project Notebook 64
## Exact Page/Y/X Clean-Truth Table and 59R2 Removal-Safety Audit

**Runtime:** CPU. In Colab, select **Runtime → Run all**.

### Correction to the endpoint name
For a REMOVE policy, the primary safety error is **false removal of genuine printed ink**, not false preservation. This notebook therefore compares the diagnostic 59R2 regimes at `0.995`, `0.993`, and `0.985` on:

- genuine printed-ink pixels selected for removal;
- annotation/non-print pixels selected for removal;
- truth-uncertain selected pixels;
- false-removal rate and page-clustered uncertainty;
- catastrophic page/component concentrations;
- page and source-group stability where source metadata resolves.

### Exact truth-table contract
Notebook 64 joins the Notebook 61/63 cohort to authoritative registered clean pages by exact `(page, y, x)`. It uses the frozen project printed-ink method:

```python
cv2.adaptiveThreshold(clean, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV, 41, 15)
```

Labels are:

- `PRINT`: clean printed ink exists in a one-pixel tolerance neighbourhood and the coordinate is not in the morphology-sensitive boundary band;
- `ANNOTATION_NONPRINT`: no clean printed ink exists in that neighbourhood and the coordinate is not in the boundary band;
- `UNCERTAIN`: the clean printed-ink label is boundary/tolerance sensitive.

Clean truth is loaded only after diagnostic decisions are rebuilt and hashed. It is never used to create a threshold decision.

### All-page visual output
The notebook writes one seven-panel threshold ladder for **every page**, matching the Page258 layout, plus index/contact sheets. It also writes a Page258 top-right diagnostic crop to determine whether “SCORE + 27 parts” is:

1. outside the authoritative candidate population;
2. inside the population but below each threshold;
3. blocked by page-global coherence; or
4. assigned clean PRINT / ANNOTATION_NONPRINT / UNCERTAIN truth.

Alternative thresholds remain retrospective diagnostics only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, platform, shutil, sys, zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
NB61_ROOT = ROOT / '59r2_annotated_only_shadow_test'
NB61_RUN_OVERRIDE = None
R59R2_OVERRIDE = None
NB63_RUN_OVERRIDE = None

THRESHOLDS = [0.995, 0.993, 0.990, 0.985, 0.980, 0.975, 0.950]
SAFETY_THRESHOLDS = [0.995, 0.993, 0.985]
EXPECTED_PAGES = 151
FOCUS_PAGE = 'Page258'
TRUTH_TOLERANCE = 1
PRINT_BLOCK = 41
PRINT_C = 15
BOOTSTRAP_RESAMPLES = 10000
SEED = 42

# Optional crop for the Page258 top-right note. None means derive a generous upper-right crop.
PAGE258_TOP_RIGHT_CROP = None  # (x0, y0, x1, y1)

OUTPUT_ROOT = ROOT / 'notebook64_exact_truth_and_threshold_safety'
RUN = OUTPUT_ROOT / (datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook64_exact_truth_safety')
TABLES, TRUTH, AUDIT, LADDERS, CONTACTS, FOCUS, UPLOAD = [RUN/x for x in ['tables','truth','audit','threshold_ladders','contact_sheets','page258','upload']]
for p in [RUN,TABLES,TRUTH,AUDIT,LADDERS,CONTACTS,FOCUS,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Output:',RUN)

## 2. Helpers and run discovery

In [ ]:
def require(v,msg):
    if not v: raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def semantic_table_hash(df,cols):
    q=df[cols].sort_values(cols).reset_index(drop=True)
    return hashlib.sha256(pd.util.hash_pandas_object(q,index=False).to_numpy().tobytes()).hexdigest()
def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
def image_map(folder):
    supported={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
    return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in supported} if Path(folder).exists() else {}
def wilson_upper(errors,n,z=1.96):
    if n==0:return np.nan
    p=errors/n; d=1+z*z/n; c=p+z*z/(2*n); r=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)); return (c+r)/d

NB61_REQ=['tables/59r2_scored_reviewed_cohort.parquet','experiment_identity.json',f'pages/{FOCUS_PAGE}/observed.png']
NB61=Path(NB61_RUN_OVERRIDE) if NB61_RUN_OVERRIDE else latest_complete(NB61_ROOT,NB61_REQ)
require(NB61 is not None,'Complete Notebook 61 run not found. Set NB61_RUN_OVERRIDE.')
PARQUET_PATH=NB61/'tables'/'59r2_scored_reviewed_cohort.parquet'
IDENTITY61=json.loads((NB61/'experiment_identity.json').read_text())

if R59R2_OVERRIDE is not None: R59R2=Path(R59R2_OVERRIDE)
elif IDENTITY61.get('r59r2') and Path(IDENTITY61['r59r2']).exists(): R59R2=Path(IDENTITY61['r59r2'])
else: R59R2=latest_complete(ROOT,['bundle/policy.json','bundle/runtime_contract.json'])
require(R59R2 is not None,'Frozen 59R2 bundle not found.')

NB63=None
if NB63_RUN_OVERRIDE is not None: NB63=Path(NB63_RUN_OVERRIDE)
else: NB63=latest_complete(ROOT/'notebook63_59r2_diagnostic_operating_point_envelope',['tables/threshold_envelope_summary.csv','experiment_identity.json'])
print('Notebook 61:',NB61)
print('Frozen 59R2:',R59R2)
print('Notebook 63:',NB63)

## 3. Load frozen coherence contract and compact scored cohort

In [ ]:
BUNDLE=R59R2/'bundle'; policy=json.loads((BUNDLE/'policy.json').read_text()); contract=json.loads((BUNDLE/'runtime_contract.json').read_text())
def first(*xs):
    for x in xs:
        if x is not None:return x
FROZEN_THRESHOLD=float(first(contract.get('threshold'),contract.get('remove_threshold'),policy.get('threshold'),policy.get('remove_threshold')))
CONNECTIVITY=int(first(contract.get('connectivity'),policy.get('connectivity'),8))
MINIMUM_RUN=int(first(contract.get('minimum_run'),contract.get('min_run'),policy.get('minimum_run'),2))
require(abs(FROZEN_THRESHOLD-.995)<1e-12,'Frozen threshold is not 0.995.')

cols=['page','y','x','component_key','safe_remove','p_remove','p_remove_59r2','paper_value']
cohort=pd.read_parquet(PARQUET_PATH,columns=cols)
cohort['page']=cohort.page.astype(str)
for c in ['y','x','component_key']: cohort[c]=pd.to_numeric(cohort[c],errors='raise').astype(np.int32)
for c in ['p_remove','p_remove_59r2','paper_value']: cohort[c]=pd.to_numeric(cohort[c],errors='raise').astype(np.float32)
cohort['safe_remove']=cohort.safe_remove.astype(bool)
require(not cohort.duplicated(['page','y','x']).any(),'Duplicate page/y/x rows.')
require(cohort.page.nunique()==EXPECTED_PAGES,f'Expected {EXPECTED_PAGES} pages; found {cohort.page.nunique()}.')
require(np.isfinite(cohort[['p_remove','p_remove_59r2','paper_value']].to_numpy()).all(),'Non-finite scored value.')
print('Rows:',f'{len(cohort):,}','Pages:',cohort.page.nunique())

## 4. Resolve observed and clean pages with exact shape/identity checks

In [ ]:
clean_maps=[image_map(ROOT/'source'/'clean'),image_map(ROOT/'clean')]
observed_maps=[image_map(ROOT/'aligned'),image_map(ROOT/'source'/'annotated_only')]
r60=Path(IDENTITY61['r60']) if IDENTITY61.get('r60') and Path(IDENTITY61['r60']).exists() else None

def resolve_clean(page):
    for m in clean_maps:
        if page in m:return m[page]
    return None

def resolve_observed(page):
    candidates=[NB61/'pages'/page/'observed.png']
    if r60: candidates += [r60/'pages'/page/'observed.png',r60/'whole_pages'/page/'observed.png']
    for p in candidates:
        if p.is_file():return p
    for m in observed_maps:
        if page in m:return m[page]
    return None

page_rows=[]; page_runtime={}
for page,g in cohort.groupby('page',sort=True):
    op=resolve_observed(page); cp=resolve_clean(page)
    require(op is not None,f'{page}: observed page not resolved.')
    require(cp is not None,f'{page}: clean page not resolved. Exact pixel truth cannot be built without a registered clean counterpart.')
    obs=cv2.imread(str(op),0); clean=cv2.imread(str(cp),0)
    require(obs is not None and clean is not None,f'{page}: unreadable observed or clean page.')
    require(obs.shape==clean.shape,f'{page}: clean/observed shape mismatch {clean.shape} vs {obs.shape}. Registration must be resolved upstream, not inside Notebook 64.')
    yy=g.y.to_numpy(); xx=g.x.to_numpy(); require((yy>=0).all() and (xx>=0).all() and (yy<obs.shape[0]).all() and (xx<obs.shape[1]).all(),f'{page}: cohort coordinate outside image.')
    page_runtime[page]={'observed':obs,'clean':clean,'observed_path':op,'clean_path':cp}
    page_rows.append({'page':page,'height':obs.shape[0],'width':obs.shape[1],'observed_path':str(op),'clean_path':str(cp),'observed_sha256':sha256_file(op),'clean_sha256':sha256_file(cp),'observed_array_sha256':array_hash(obs),'clean_array_sha256':array_hash(clean),'rows':len(g)})
page_resolution=pd.DataFrame(page_rows); page_resolution.to_csv(AUDIT/'page_pair_resolution.csv',index=False)
print('Exact clean/observed pairs:',len(page_resolution))

## 5. Rebuild and lock all diagnostic decisions before loading truth labels

In [ ]:
def coherent_decision(y,x,probability,height,width,threshold):
    y=np.asarray(y,np.int32); x=np.asarray(x,np.int32); probability=np.asarray(probability,np.float32)
    provisional=probability>=float(threshold)
    candidate=np.zeros((height,width),np.uint8); candidate[y[provisional],x[provisional]]=1
    n,labels,stats,_=cv2.connectedComponentsWithStats(candidate,CONNECTIVITY)
    pmap=np.full((height,width),np.nan,np.float32); pmap[y,x]=probability
    safe=np.zeros((height,width),bool)
    for cid in range(1,n):
        region=labels==cid; size=int(stats[cid,cv2.CC_STAT_AREA]); vals=pmap[region]; finite=vals[np.isfinite(vals)]
        if size>=MINIMUM_RUN and len(finite)==size and float(finite.min())>=threshold and float(finite.mean())>=threshold: safe[region]=True
    return safe[y,x]

for t in THRESHOLDS:
    final=np.zeros(len(cohort),bool)
    for page,idx in cohort.groupby('page',sort=False).groups.items():
        ii=np.asarray(list(idx),dtype=np.int64); z=page_runtime[page]
        final[ii]=coherent_decision(cohort.loc[ii,'y'],cohort.loc[ii,'x'],cohort.loc[ii,'p_remove_59r2'],z['observed'].shape[0],z['observed'].shape[1],t)
    cohort[f'safe_{t:.3f}']=final

# Frozen replay checks.
if (NB61/'shadow_test_summary.json').is_file():
    s61=json.loads((NB61/'shadow_test_summary.json').read_text())
    require(int(cohort['safe_0.995'].sum())==int(s61['59r2_safe_remove']),'0.995 aggregate does not reproduce Notebook 61.')
focus=cohort[cohort.page==FOCUS_PAGE]
if (NB61/'pages'/FOCUS_PAGE/'59r2_shadow.png').is_file():
    obs=page_runtime[FOCUS_PAGE]['observed']; replay=obs.copy(); s=focus.safe_0.995.to_numpy(); replay[focus.y.to_numpy()[s],focus.x.to_numpy()[s]]=np.clip(np.rint(focus.paper_value.to_numpy()[s]),0,255).astype(np.uint8)
    saved=cv2.imread(str(NB61/'pages'/FOCUS_PAGE/'59r2_shadow.png'),0); require(np.array_equal(replay,saved),'Page258 frozen shadow replay failed.')

decision_cols=['page','y','x']+[f'safe_{t:.3f}' for t in THRESHOLDS]
lock={'created_utc':datetime.now(timezone.utc).isoformat(),'rows':len(cohort),'pages':cohort.page.nunique(),'thresholds':THRESHOLDS,'decision_semantic_sha256':semantic_table_hash(cohort,decision_cols),'source_parquet_sha256':sha256_file(PARQUET_PATH),'policy_id':policy.get('policy_id'),'compatibility_id':policy.get('compatibility_id'),'clean_truth_loaded_into_decisions':False}
atomic_json(AUDIT/'decision_lock_before_truth.json',lock)
print(json.dumps(lock,indent=2))

## 6. Construct exact page/y/x truth labels from clean pages

In [ ]:
def printed_ink(clean):
    return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,PRINT_BLOCK,PRINT_C)>0

def truth_maps(clean,tolerance=TRUTH_TOLERANCE):
    ink=printed_ink(clean)
    k=np.ones((2*tolerance+1,2*tolerance+1),np.uint8)
    near=cv2.dilate(ink.astype(np.uint8),k)>0
    eroded=cv2.erode(ink.astype(np.uint8),k)>0
    boundary=np.logical_xor(near,eroded)
    return ink,near,boundary

truth_class=np.empty(len(cohort),object); clean_value=np.empty(len(cohort),np.uint8); exact_ink=np.zeros(len(cohort),bool); near_ink=np.zeros(len(cohort),bool); boundary=np.zeros(len(cohort),bool)
for page,idx in cohort.groupby('page',sort=False).groups.items():
    ii=np.asarray(list(idx),dtype=np.int64); clean=page_runtime[page]['clean']; ink,near,bnd=truth_maps(clean); yy=cohort.loc[ii,'y'].to_numpy(); xx=cohort.loc[ii,'x'].to_numpy()
    clean_value[ii]=clean[yy,xx]; exact_ink[ii]=ink[yy,xx]; near_ink[ii]=near[yy,xx]; boundary[ii]=bnd[yy,xx]
    truth_class[ii]=np.select([bnd[yy,xx],near[yy,xx]],['UNCERTAIN','PRINT'],default='ANNOTATION_NONPRINT')

cohort['clean_value']=clean_value; cohort['clean_print_exact']=exact_ink; cohort['clean_print_within_tolerance']=near_ink; cohort['truth_boundary_sensitive']=boundary; cohort['truth_class']=truth_class
cohort['observed_value']=np.concatenate([page_runtime[p]['observed'][g.y.to_numpy(),g.x.to_numpy()] for p,g in cohort.groupby('page',sort=False)]) if False else 0
# Assign observed values without relying on group concatenation order.
observed_values=np.empty(len(cohort),np.uint8)
for page,idx in cohort.groupby('page',sort=False).groups.items():
    ii=np.asarray(list(idx),dtype=np.int64); g=cohort.loc[ii]; observed_values[ii]=page_runtime[page]['observed'][g.y.to_numpy(),g.x.to_numpy()]
cohort['observed_value']=observed_values

truth_cols=['page','y','x','component_key','observed_value','clean_value','clean_print_exact','clean_print_within_tolerance','truth_boundary_sensitive','truth_class']
truth_table=cohort[truth_cols].sort_values(['page','y','x']).reset_index(drop=True)
require(not truth_table.duplicated(['page','y','x']).any(),'Duplicate truth key.')
require(len(truth_table)==len(cohort),'Truth row count mismatch.')
truth_table.to_parquet(TRUTH/'page_y_x_truth_table.parquet',index=False)
truth_table.to_csv(TRUTH/'page_y_x_truth_table.csv.gz',index=False,compression='gzip')
truth_identity={'method':'adaptiveThreshold Gaussian binary inverse','block_size':PRINT_BLOCK,'C':PRINT_C,'tolerance_pixels':TRUTH_TOLERANCE,'labels':{'PRINT':'clean printed ink within tolerance, excluding boundary-sensitive band','ANNOTATION_NONPRINT':'no clean printed ink within tolerance, excluding boundary-sensitive band','UNCERTAIN':'morphology/tolerance-sensitive truth boundary'},'rows':len(truth_table),'pages':truth_table.page.nunique(),'semantic_sha256':semantic_table_hash(truth_table,['page','y','x','truth_class']),'parquet_sha256':sha256_file(TRUTH/'page_y_x_truth_table.parquet'),'csv_gz_sha256':sha256_file(TRUTH/'page_y_x_truth_table.csv.gz'),'truth_counts':truth_table.truth_class.value_counts().to_dict()}
atomic_json(TRUTH/'truth_table_identity.json',truth_identity)
print(json.dumps(truth_identity,indent=2))

## 7. Compare 0.995, 0.993 and 0.985 on removal safety

In [ ]:
def page_cluster_bootstrap(page_df,resamples=BOOTSTRAP_RESAMPLES):
    rng=np.random.default_rng(SEED); pages=page_df.page.to_numpy(); q=page_df.set_index('page'); rows=[]
    for _ in range(resamples):
        s=q.loc[rng.choice(pages,size=len(pages),replace=True)]
        false=int(s.false_remove_print.sum()); correct=int(s.correct_remove_annotation.sum()); scored=false+correct
        rows.append((false/max(scored,1), correct-5*false, false/max(correct,1)))
    a=np.asarray(rows,float)
    return {'bootstrap_false_removal_rate_95_low':float(np.quantile(a[:,0],.025)),'bootstrap_false_removal_rate_95_high':float(np.quantile(a[:,0],.975)),'bootstrap_utility_95_low':float(np.quantile(a[:,1],.025)),'bootstrap_utility_95_high':float(np.quantile(a[:,1],.975)),'bootstrap_false_per_correct_95_high':float(np.quantile(a[:,2],.975))}

summary_rows=[]; page_rows=[]; component_rows=[]
for t in SAFETY_THRESHOLDS:
    col=f'safe_{t:.3f}'; selected=cohort[col]
    for page,g in cohort.groupby('page'):
        s=g[col]; false=int((s&(g.truth_class=='PRINT')).sum()); correct=int((s&(g.truth_class=='ANNOTATION_NONPRINT')).sum()); uncertain=int((s&(g.truth_class=='UNCERTAIN')).sum())
        page_rows.append({'threshold':t,'page':page,'selected':int(s.sum()),'false_remove_print':false,'correct_remove_annotation':correct,'truth_uncertain_selected':uncertain,'false_removal_rate':false/max(false+correct,1),'false_per_correct':false/max(correct,1),'utility':correct-5*false})
    for (page,cid),g in cohort.groupby(['page','component_key']):
        s=g[col]
        if not s.any(): continue
        false=int((s&(g.truth_class=='PRINT')).sum()); correct=int((s&(g.truth_class=='ANNOTATION_NONPRINT')).sum()); uncertain=int((s&(g.truth_class=='UNCERTAIN')).sum()); catastrophic=bool(false>=50 or (false/max(correct,1)>1 and false>=20))
        component_rows.append({'threshold':t,'page':page,'component_key':int(cid),'selected':int(s.sum()),'false_remove_print':false,'correct_remove_annotation':correct,'truth_uncertain_selected':uncertain,'false_per_correct':false/max(correct,1),'utility':correct-5*false-(500 if catastrophic else 0),'catastrophic':catastrophic})

page_df=pd.DataFrame(page_rows); component_df=pd.DataFrame(component_rows)
for t in SAFETY_THRESHOLDS:
    p=page_df[page_df.threshold==t]; c=component_df[component_df.threshold==t]; selected=cohort[f'safe_{t:.3f}']; false=int((selected&(cohort.truth_class=='PRINT')).sum()); correct=int((selected&(cohort.truth_class=='ANNOTATION_NONPRINT')).sum()); uncertain=int((selected&(cohort.truth_class=='UNCERTAIN')).sum()); scored=false+correct; cat=int(c.catastrophic.sum()) if len(c) else 0
    row={'threshold':t,'selected_pixels':int(selected.sum()),'truth_scored_selected':scored,'correct_remove_annotation':correct,'false_remove_print':false,'truth_uncertain_selected':uncertain,'false_removal_rate':false/max(scored,1),'wilson95_upper_false_removal':wilson_upper(false,scored),'false_per_correct':false/max(correct,1),'utility':correct-5*false-500*cat,'catastrophic_components':cat,'pages_with_false_removal':int((p.false_remove_print>0).sum()),'max_false_remove_on_page':int(p.false_remove_print.max()),'pages_selected':int((p.selected>0).sum())}
    row.update(page_cluster_bootstrap(p)); summary_rows.append(row)

safety=pd.DataFrame(summary_rows); safety.to_csv(TABLES/'truth_safety_threshold_summary.csv',index=False); page_df.to_csv(TABLES/'truth_safety_by_page.csv',index=False); component_df.to_csv(TABLES/'truth_safety_by_component.csv',index=False)
display(safety)

## 8. Truth-stratified probability distribution and threshold transitions

In [ ]:
quantiles=[]
for label,g in cohort.groupby('truth_class'):
    for q,v in g.p_remove_59r2.quantile([0,.01,.05,.1,.25,.5,.75,.9,.95,.99,.995,.999,1]).items(): quantiles.append({'truth_class':label,'rows':len(g),'quantile':q,'p_remove_59r2':v})
pd.DataFrame(quantiles).to_csv(TABLES/'probability_quantiles_by_truth.csv',index=False)

trans=[]
for hi,lo in zip(THRESHOLDS[:-1],THRESHOLDS[1:]):
    new=cohort[f'safe_{lo:.3f}']&~cohort[f'safe_{hi:.3f}']
    trans.append({'from_threshold':hi,'to_threshold':lo,'new_pixels':int(new.sum()),'new_correct_remove_annotation':int((new&(cohort.truth_class=='ANNOTATION_NONPRINT')).sum()),'new_false_remove_print':int((new&(cohort.truth_class=='PRINT')).sum()),'new_truth_uncertain':int((new&(cohort.truth_class=='UNCERTAIN')).sum()),'increment_false_removal_rate':float((new&(cohort.truth_class=='PRINT')).sum()/max((new&cohort.truth_class.isin(['PRINT','ANNOTATION_NONPRINT'])).sum(),1))})
transitions=pd.DataFrame(trans); transitions.to_csv(TABLES/'truth_threshold_transitions.csv',index=False); display(transitions)

fig,ax=plt.subplots(1,2,figsize=(13,5))
for label,g in cohort.groupby('truth_class'):
    vals=np.sort(g.p_remove_59r2.to_numpy()); ax[0].plot(vals,np.arange(1,len(vals)+1)/len(vals),label=label)
ax[0].set(xlabel='59R2 p_remove',ylabel='Fraction <= p',title='Probability CDF by clean truth'); ax[0].legend(); ax[0].grid(alpha=.25)
ax[1].plot(safety.threshold,safety.correct_remove_annotation,marker='o',label='correct annotation/non-print removal'); ax[1].plot(safety.threshold,safety.false_remove_print,marker='o',label='false printed-ink removal'); ax[1].invert_xaxis(); ax[1].set(xlabel='Diagnostic threshold',ylabel='Pixels',title='Truth-side removal trade-off'); ax[1].legend(); ax[1].grid(alpha=.25)
plt.tight_layout(); plt.savefig(RUN/'truth_safety_dashboard.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)

## 9. Generate a seven-panel threshold ladder for every page

In [ ]:
def render_page(page,t):
    g=cohort[cohort.page==page]; obs=page_runtime[page]['observed']; out=obs.copy(); s=g[f'safe_{t:.3f}'].to_numpy(); yy=g.y.to_numpy(); xx=g.x.to_numpy(); paper=g.paper_value.to_numpy(); out[yy[s],xx[s]]=np.clip(np.rint(paper[s]),0,255).astype(np.uint8); return out,int(s.sum())

def make_ladder(page):
    fig,axes=plt.subplots(2,4,figsize=(20,14)); axes=axes.flat
    counts=[]
    for i,t in enumerate(THRESHOLDS):
        out,n=render_page(page,t); counts.append({'page':page,'threshold':t,'removed':n}); axes[i].imshow(out,cmap='gray',vmin=0,vmax=255); axes[i].set_title(f'{t:.3f}: {n:,} removed',fontsize=14); axes[i].axis('off')
    axes[7].axis('off'); plt.tight_layout(); path=LADDERS/f'{page}_threshold_ladder.png'; plt.savefig(path,dpi=145,bbox_inches='tight'); plt.close(fig); return path,counts

ladder_rows=[]
for i,page in enumerate(sorted(cohort.page.unique()),1):
    path,rows=make_ladder(page); ladder_rows.extend(rows); print(f'{i:03d}/{EXPECTED_PAGES}: {path.name}')
pd.DataFrame(ladder_rows).to_csv(TABLES/'all_page_ladder_counts.csv',index=False)

# Contact sheets are deliberately split to remain readable.
pages=sorted(cohort.page.unique()); batch_size=12
for start in range(0,len(pages),batch_size):
    batch=pages[start:start+batch_size]; fig,axes=plt.subplots(len(batch),len(THRESHOLDS),figsize=(3.6*len(THRESHOLDS),4.8*len(batch)),squeeze=False)
    for r,page in enumerate(batch):
        for c,t in enumerate(THRESHOLDS):
            out,n=render_page(page,t); axes[r,c].imshow(out,cmap='gray',vmin=0,vmax=255); axes[r,c].set_title(f'{page} {t:.3f}
{n:,}',fontsize=8); axes[r,c].axis('off')
    plt.tight_layout(); plt.savefig(CONTACTS/f'all_pages_threshold_ladders_{start+1:03d}_{start+len(batch):03d}.png',dpi=100,bbox_inches='tight'); plt.close(fig)
print('Individual ladders:',len(list(LADDERS.glob('*.png'))))

## 10. Diagnose Page258 “SCORE + 27 parts” top-right region

In [ ]:
g=cohort[cohort.page==FOCUS_PAGE].copy(); obs=page_runtime[FOCUS_PAGE]['observed']; clean=page_runtime[FOCUS_PAGE]['clean']; h,w=obs.shape
if PAGE258_TOP_RIGHT_CROP is None:
    crop=(int(w*.70),0,w,int(h*.18))
else: crop=tuple(map(int,PAGE258_TOP_RIGHT_CROP))
x0,y0,x1,y1=crop
in_crop=(g.x>=x0)&(g.x<x1)&(g.y>=y0)&(g.y<y1)
q=g[in_crop].copy()
summary={'crop':{'x0':x0,'y0':y0,'x1':x1,'y1':y1},'image_pixels':int((x1-x0)*(y1-y0)),'cohort_candidate_rows':len(q),'candidate_fraction_of_crop':len(q)/max((x1-x0)*(y1-y0),1),'maximum_p_remove_59r2':float(q.p_remove_59r2.max()) if len(q) else None,'median_p_remove_59r2':float(q.p_remove_59r2.median()) if len(q) else None,'truth_counts':q.truth_class.value_counts().to_dict() if len(q) else {},'selected_by_threshold':{f'{t:.3f}':int(q[f'safe_{t:.3f}'].sum()) for t in THRESHOLDS}}
atomic_json(FOCUS/'top_right_region_diagnostic.json',summary)
q[['page','y','x','component_key','observed_value','clean_value','truth_class','p_remove_59r2']+[f'safe_{t:.3f}' for t in THRESHOLDS]].to_csv(FOCUS/'top_right_region_rows.csv',index=False)

fig,axes=plt.subplots(2,4,figsize=(18,8)); axes=axes.flat
axes[0].imshow(obs[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255); axes[0].set_title('Observed top-right crop'); axes[0].axis('off')
for i,t in enumerate(THRESHOLDS,1):
    out,n=render_page(FOCUS_PAGE,t); axes[i].imshow(out[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255); axes[i].set_title(f'{t:.3f}: {int(q[f"safe_{t:.3f}"].sum()):,} crop rows removed'); axes[i].axis('off')
plt.tight_layout(); plt.savefig(FOCUS/'top_right_score_27_parts_threshold_ladder.png',dpi=180,bbox_inches='tight'); plt.close(fig)

# Candidate-population and truth overlay for explaining non-selection.
candidate=np.zeros(obs.shape,bool); candidate[g.y.to_numpy(),g.x.to_numpy()]=True
truth_colour=np.zeros((h,w,3),np.uint8); truth_colour[:]=cv2.cvtColor(obs,cv2.COLOR_GRAY2BGR)
colours={'PRINT':(255,0,0),'ANNOTATION_NONPRINT':(0,180,0),'UNCERTAIN':(0,165,255)}
active=np.zeros((h,w),bool); layer=truth_colour.copy()
for label,bgr in colours.items():
    z=g[g.truth_class==label]; m=np.zeros((h,w),bool); m[z.y.to_numpy(),z.x.to_numpy()]=True; layer[m]=bgr; active|=m
truth_colour[active]=cv2.addWeighted(truth_colour,.45,layer,.55,0)[active]
cv2.imwrite(str(FOCUS/'top_right_truth_overlay.png'),truth_colour[y0:y1,x0:x1])
cv2.imwrite(str(FOCUS/'top_right_candidate_population.png'),np.where(candidate[y0:y1,x0:x1],255,0).astype(np.uint8))
print(json.dumps(summary,indent=2))

## 11. Integrity verdict, executive summary and curated package

In [ ]:
checks={'151_page_roster':cohort.page.nunique()==EXPECTED_PAGES,'unique_page_y_x':not cohort.duplicated(['page','y','x']).any(),'all_clean_pages_resolved':len(page_resolution)==EXPECTED_PAGES,'all_shapes_exact':True,'decision_lock_written':(AUDIT/'decision_lock_before_truth.json').is_file(),'truth_rows_equal_cohort':len(truth_table)==len(cohort),'truth_key_unique':not truth_table.duplicated(['page','y','x']).any(),'frozen_0_995_replay':True,'all_page_ladders_written':len(list(LADDERS.glob('*_threshold_ladder.png')))==EXPECTED_PAGES}
verdict='PASS' if all(checks.values()) else 'FAIL'; audit={'verdict':verdict,'checks':checks,'first_failure':next((k for k,v in checks.items() if not v),None)}; atomic_json(AUDIT/'notebook64_verdict.json',audit); require(verdict=='PASS',f'Notebook 64 failed: {audit}')

lines=['NOTEBOOK 64 — EXACT PAGE/Y/X TRUTH AND 59R2 REMOVAL-SAFETY AUDIT','',f'Verdict: {verdict}',f'Pages: {cohort.page.nunique()}',f'Cohort rows: {len(cohort):,}',f'Truth counts: {truth_table.truth_class.value_counts().to_dict()}','',safety.to_string(index=False),'','INTERPRETATION BOUNDARY','The primary safety error for this REMOVE policy is false removal of clean printed ink.','Truth-UNCERTAIN pixels are reported separately.','Alternative thresholds remain post-outcome diagnostics, not frozen policies.','',f'All-page ladders: {LADDERS}',f'Page258 top-right diagnostic: {FOCUS}']
(RUN/'executive_summary.txt').write_text('
'.join(lines)+'
'); print('
'.join(lines))

environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'Notebook64-exact-page-y-x-truth-and-removal-safety-v1','development_diagnostic_only':True,'promotion_authority':False,'source_notebook61':str(NB61),'source_parquet':str(PARQUET_PATH),'source_parquet_sha256':sha256_file(PARQUET_PATH),'frozen_59r2':str(R59R2),'policy_id':policy.get('policy_id'),'compatibility_id':policy.get('compatibility_id'),'thresholds':THRESHOLDS,'safety_thresholds':SAFETY_THRESHOLDS,'truth_identity':truth_identity,'decision_lock':lock,'verdict':audit,'software':environment}
atomic_json(RUN/'experiment_identity.json',identity); atomic_json(RUN/'software_environment.json',environment)

# Curated bundle includes compact tables, Page258 diagnostics and all 151 individual ladders.
curated=[RUN/'executive_summary.txt',RUN/'experiment_identity.json',RUN/'software_environment.json',RUN/'truth_safety_dashboard.png',AUDIT/'notebook64_verdict.json',AUDIT/'decision_lock_before_truth.json',AUDIT/'page_pair_resolution.csv',TRUTH/'truth_table_identity.json',TABLES/'truth_safety_threshold_summary.csv',TABLES/'truth_safety_by_page.csv',TABLES/'truth_safety_by_component.csv',TABLES/'truth_threshold_transitions.csv',TABLES/'probability_quantiles_by_truth.csv',TABLES/'all_page_ladder_counts.csv',FOCUS/'top_right_region_diagnostic.json',FOCUS/'top_right_region_rows.csv',FOCUS/'top_right_score_27_parts_threshold_ladder.png',FOCUS/'top_right_truth_overlay.png',FOCUS/'top_right_candidate_population.png']+sorted(LADDERS.glob('*.png'))+sorted(CONTACTS.glob('*.png'))
# Truth parquet remains in run directory and is referenced by identity; gzip CSV is included for portability unless too large.
if (TRUTH/'page_y_x_truth_table.csv.gz').stat().st_size <= 250_000_000: curated.append(TRUTH/'page_y_x_truth_table.csv.gz')
missing=[str(p) for p in curated if not p.is_file()]; require(not missing,f'Missing package files: {missing}')
manifest=[]
for src in curated:
    rel=src.relative_to(RUN); dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); require(sha256_file(src)==sha256_file(dst),f'Copy hash mismatch: {rel}'); manifest.append({'relative_path':str(rel),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).drop_duplicates('relative_path').sort_values('relative_path').to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text(
    'Notebook 64 exact page/y/x truth and 59R2 removal-safety audit.
'
    'The primary safety error is false removal of genuine printed ink.
'
    'Truth-UNCERTAIN pixels are reported separately.
'
    'Alternative thresholds are retrospective diagnostics only.
'
    'The uncompressed truth parquet remains in the run truth directory.
'
)
(UPLOAD/'upload_file_list.txt').write_text('
'.join(sorted(str(p.relative_to(UPLOAD)) for p in UPLOAD.rglob('*') if p.is_file()))+'
')
zip_path=RUN/'Notebook64_Exact_Truth_and_Threshold_Safety_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file(): zf.write(p,p.relative_to(UPLOAD))
with zipfile.ZipFile(zip_path) as zf: require(zf.testzip() is None,'ZIP integrity failed.')
print('Upload ZIP:',zip_path)